In [ ]:
# === DATA IMPORT: corrected Ge IQ files ===
import os, re
import numpy as np

DATA_FILE = ""                #insert your filename ().npy) here
raw = np.load(DATA_FILE, allow_pickle=True)  
if len(raw) == 0:
    raise ValueError(f'{DATA_FILE} contains no files.')
if not all(isinstance(r, dict) and {'filename', 'data', 'fs'} <= set(r) for r in raw):
    raise ValueError("Expected the fields for each entry: 'filename', 'data', and 'fs'.")

fs_values = np.asarray([float(r['fs']) for r in raw])
sample_counts = np.asarray([len(r['data']) for r in raw], int)
if not np.allclose(fs_values, fs_values[0]):
    raise ValueError('The sampling rates of the files are not identical.')
if np.any(sample_counts != sample_counts[0]):
    raise ValueError('The number of IQ samples is not identical in all files.')

fs = float(fs_values[0])
n_samp = int(sample_counts[0])
iq_all = np.stack([np.asarray(r['data'], np.complex64) for r in raw])
source_files = [str(r['filename']) for r in raw]
names = np.asarray([os.path.basename(p) for p in source_files])
file_paths = list(source_files)

def _ts(path):
    m = re.search(r'(\d{4}\.\d{2}\.\d{2}\.\d{2}\.\d{2}\.\d{2}\.\d{3})', str(path))
    return m.group(1) if m else ''

timestamps = [_ts(p) for p in source_files]
if all(timestamps) and timestamps != sorted(timestamps):
    order = np.argsort(timestamps)
    iq_all = iq_all[order]
    names = names[order]
    source_files = [source_files[i] for i in order]
    file_paths = list(source_files)
    timestamps = [timestamps[i] for i in order]

N = NF = len(iq_all)
dur = n_samp / fs
print(f'DONE: iq_all {iq_all.shape} ({iq_all.nbytes/1e6:.1f} MB, {iq_all.dtype})')
print(f'fs={fs:.3f} Hz | duration/file={dur:.6f} s | {N} Files')
if all(timestamps):
    print(f'Time range: {timestamps[0]} .. {timestamps[-1]}')

In [ ]:
# === STFT AND CENTERING: generates the inputs for the staircase fit ===
import numpy as np
from scipy.signal import stft
from scipy.ndimage import median_filter

DT_BIN = 0.01       # desired time step
NPERSEG = 500       
hop = int(round(DT_BIN * fs))
noverlap = NPERSEG - hop
if noverlap < 0:
    raise ValueError(f'NPERSEG={NPERSEG} < hop={hop}; NPERSEG must be at least {hop} for 10 ms bins.')

f0, t0, _ = stft(iq_all[0], fs=fs, nperseg=NPERSEG, noverlap=noverlap,
                 return_onesided=False, boundary=None, padded=False)
base_freq = np.fft.fftshift(f0)
times = np.asarray(t0)
n_freq, n_time = len(base_freq), len(times)
df = fs / NPERSEG

summiert = np.empty((NF, n_freq, n_time), dtype=np.float32)
for i in range(NF):
    _, _, Z = stft(iq_all[i], fs=fs, nperseg=NPERSEG, noverlap=noverlap,
                   return_onesided=False, boundary=None, padded=False)
    summiert[i] = (np.abs(np.fft.fftshift(Z, axes=0))**2).astype(np.float32)


raw_gbin = np.asarray([np.argmax(s.sum(axis=1)) for s in summiert], int)
gbin = median_filter(raw_gbin, size=11, mode='nearest')
center = n_freq // 2
f_ground_per_file = base_freq[gbin]
CENTERING_DELTA_BINS = raw_gbin - gbin
CENTERING_SUSPECT_IDX = np.flatnonzero(np.abs(CENTERING_DELTA_BINS) > 1)
for i in range(NF):
    summiert[i] = np.roll(summiert[i], center-int(gbin[i]), axis=0)
frequency = (np.arange(n_freq)-center) * df

summiert_full, times_full = summiert, times
dt_e = float(times[1]-times[0])
print(f'df={df:.2f} Hz | window={NPERSEG/fs*1e3:.3f} ms | dt(bin)={dt_e*1e3:.3f} ms')
print(f'summiert {summiert.shape} ({summiert.nbytes/1e6:.1f} MB, {summiert.dtype})')
print(f'Ground-state frequency relative to the recorder: {f_ground_per_file.min():.1f} .. {f_ground_per_file.max():.1f} Hz')
print(f'Centering QC: {len(CENTERING_SUSPECT_IDX)} files with |raw_gbin-gbin| > 1')

In [ ]:
# === FREQUENCY BAND CHECK ===
import matplotlib.pyplot as plt
# y = summed power, x = frequency relative to the ground state. Shows the ground (0 Hz) and excited states.
BAND_E   = (-2650, -1600)                      # <<< excited state (after centering)
BG_BANDS = [(-5000, -3500), (-3500, -3000)]    # <<< flat background (for the later background correction)

A1_EXTERNAL = 0.81 # only important if USE_CALIBRATED_A1 = False and a preset of Amplitude is wanted

prof = summiert.sum(axis=(0, 2))               # (freq,) - sum over files and time

plt.figure(figsize=(11, 5))
plt.semilogy(frequency, prof, color="steelblue")
plt.axvline(0, color="k", ls=":", lw=0.9, label="Ground state (0 Hz)")
plt.axvspan(BAND_E[0], BAND_E[1], color="tomato", alpha=0.25, label=f"excited {BAND_E}")
for _j, (lo, hi) in enumerate(BG_BANDS):
    plt.axvspan(lo, hi, color="green", alpha=0.18, label="Background (bg)" if _j == 0 else None)
plt.xlim(-3000, -1500)
plt.xlabel("Frequency relative to ground state [Hz]")
plt.ylabel("Σ power |Z|² (all files and times)")
plt.title(f"Ge - summed frequency profile ({NF} Files)")
plt.legend(); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

In [ ]:
# === BAND INTEGRATION AND NOISE NORMALIZATION FOR THE STAIRCASE FIT ===
mb_e = (frequency > BAND_E[0]) & (frequency < BAND_E[1])
mfar_e = np.zeros(len(frequency), bool)
for _lo, _hi in BG_BANDS:
    mfar_e |= (frequency > _lo) & (frequency < _hi)
if not np.any(mb_e):
    raise ValueError(f'BAND_E={BAND_E} contains no frequency bin.')
if not np.any(mfar_e):
    raise ValueError(f'BG_BANDS={BG_BANDS} contain no frequency bin.')
if np.any(mb_e & mfar_e):
    raise ValueError('The signal and background bands overlap.')

ANALYSIS_BAND_CONFIG = {
    'signal_Hz': tuple(BAND_E),
    'background_Hz': tuple(BG_BANDS),
}
f_e, t_e, dt_e = frequency, times, float(times[1]-times[0])

Y_e = summiert[:, mb_e, :].mean(axis=1)
bg_e = summiert[:, mfar_e, :].mean(axis=1)
y_files_e = Y_e-bg_e

# Without stable contamination, the late, stable terminal segments belong to
# the ground state. The minimum count scales with the data-set size so this data
# block also works with substantially fewer files.
_REF_SIGMA = 0.66
_NOISE_BINS = min(15, y_files_e.shape[1])
_MIN_GROUND_FILES = min(
    len(y_files_e), max(3, min(20, int(np.ceil(0.20*len(y_files_e)))))
)
if len(y_files_e) < 3:
    raise RuntimeError('At least three files are required for the noise calibration.')

_end = np.asarray(y_files_e[:, -_NOISE_BINS:], float)
_end_mean = _end.mean(axis=1)
_end_std = _end.std(axis=1)
_valid = np.isfinite(_end_mean) & np.isfinite(_end_std) & (_end_std > 0)
if np.count_nonzero(_valid) < _MIN_GROUND_FILES:
    raise RuntimeError(
        f'Only {np.count_nonzero(_valid)} valid files; '
        f'at least {_MIN_GROUND_FILES} are required.'
    )

_h = max(2, _NOISE_BINS//3)
_end_change = np.abs(_end[:, :_h].mean(axis=1)-_end[:, -_h:].mean(axis=1))
_change_scale = float(np.median(_end_std[_valid]))
_stable = _valid & (_end_change <= 1.5*_change_scale)
if np.count_nonzero(_stable) < _MIN_GROUND_FILES:
    # For small data sets, use the most stable valid terminal segments instead of
    # failing because of an absolute count intended for large data sets.
    _order = np.flatnonzero(_valid)[np.argsort(_end_change[_valid])]
    _stable = np.zeros(len(y_files_e), bool)
    _stable[_order[:_MIN_GROUND_FILES]] = True
GROUND_NOISE_IDX = np.flatnonzero(_stable)

GROUND_SIGMA_RAW = float(np.median(_end_std[GROUND_NOISE_IDX]))
if not np.isfinite(GROUND_SIGMA_RAW) or GROUND_SIGMA_RAW <= 0:
    raise RuntimeError('Invalid ground-state noise width.')
_ground_residual = (
    _end[GROUND_NOISE_IDX]
    -_end[GROUND_NOISE_IDX].mean(axis=1, keepdims=True)
)
_acf_den = float(np.sum(_ground_residual[:, :-1]**2))
GROUND_ACF1 = (
    float(np.sum(_ground_residual[:, :-1]*_ground_residual[:, 1:])/_acf_den)
    if _acf_den > 0 else np.nan
)

y_files_e = y_files_e*(_REF_SIGMA/GROUND_SIGMA_RAW)
sigma_e = _REF_SIGMA
print(f'BAND_E={BAND_E} ({int(mb_e.sum())} Bins) | BG={int(mfar_e.sum())} Bins')
print(f'y_files_e {y_files_e.shape} (files x time) | dt_e={dt_e*1e3:.3f} ms')
print(f'Ground-state level: '
      f'{np.median(y_files_e[GROUND_NOISE_IDX, -_NOISE_BINS:].mean(axis=1)):+.3f}')
print(f'Noise normalization: {len(GROUND_NOISE_IDX)} stable terminal segments '
      f'| raw sigma={GROUND_SIGMA_RAW:.4g} -> {sigma_e}')
print(f'Ground lag-1 ACF={GROUND_ACF1:+.3f}')



In [ ]:
# === INDEPENDENT A1 CALIBRATION (RUNS BEFORE THE STAIRCASE FIT) ===


import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from scipy.optimize import brentq

# ---------------- Calibration parameters ----------------
A1_CAL_NMAX = 22          # largest ion count considered
A1_CAL_PROFILE = np.array([0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5, 1.8, 2.2])
A1_CAL_MC_REPS = 24       # parametric bootstrap for bias and uncertainty
A1_CAL_SEED = 20260828
A1_CAL_EM_ITER = 800
A1_CAL_FILE_IDX = None    # None = all files
# ------------------------------------------------------------


def _a1cal_lse(a, axis):
    mx = np.max(a, axis=axis, keepdims=True)
    return (mx + np.log(np.sum(np.exp(a-mx), axis=axis, keepdims=True))).squeeze(axis)


def _a1cal_var_model(R_N, S_N, N2, a, c, n_fp=8):
    """min_(a,c) sum_N [S_N/u_N + R_N*log u_N] with u_N = a + c*N^2 (fixed point)."""
    for _ in range(n_fp):
        u = np.maximum(a + c*N2, 1e-12)
        w = 1.0/(u*u)
        wR, wS = w*R_N, w*S_N
        M = np.array([[np.sum(wR), np.sum(wR*N2)],
                      [np.sum(wR*N2), np.sum(wR*N2*N2)]])
        rhs = np.array([np.sum(wS), np.sum(wS*N2)])
        try:
            sol = np.linalg.solve(M, rhs)
        except np.linalg.LinAlgError:
            break
        if not np.isfinite(sol).all() or sol[0] <= 0:
            break
        a, c = float(sol[0]), max(float(sol[1]), 0.0)
    return max(a, 1e-12), max(c, 0.0)


def _a1cal_exp_mu(n_t, t_s, n_files):
    """Poisson MLE for mu_t = mu0*exp(-lam*t) from expected ion counts n_t."""
    total = float(np.sum(n_t))
    if total <= 0:
        return 1e-9, 0.0
    target = float(np.sum(n_t*t_s)/total)

    def imbalance(lam):
        w = np.exp(-lam*t_s)
        return float(np.sum(t_s*w)/np.sum(w))-target

    lo, hi = 1e-4, 5.0e3
    if imbalance(lo) < 0:
        lam = lo
    elif imbalance(hi) > 0:
        lam = hi
    else:
        lam = float(brentq(imbalance, lo, hi, xtol=1e-9, rtol=1e-13, maxiter=300))
    mu0 = total/(n_files*float(np.sum(np.exp(-lam*t_s))))
    return max(mu0, 1e-9), lam


def fit_power_mixture(X, t_s, n_max=22, mode='exp', A1_fixed=None, init=None,
                      n_iter=800, tol=1e-11):
    """EM fit of the Poisson-Gaussian mixture. X: (time bins, files).

    mode='exp'  -> mu_t = mu0*exp(-lam*t), two parameters (decay law).
    mode='free' -> mu_t is free in each time bin; used only as a robustness check and
                   is substantially less well identified (see output).
    """
    X = np.asarray(X, float)
    if X.ndim != 2 or X.shape[1] < 3 or np.any(~np.isfinite(X)):
        raise ValueError('X must be a finite 2D array (time bins x files).')
    T, n_files = X.shape
    t_s = np.asarray(t_s, float)
    if t_s.shape != (T,):
        raise ValueError('t_s must be a time axis of length X.shape[0].')
    N = np.arange(int(n_max)+1, dtype=float)
    N2 = N*N
    log_fact = gammaln(N+1.0)
    m_t = X.mean(axis=1)
    init = dict(init or {})
    A1 = float(init.get('A1', max(1e-3, 0.25*(np.max(m_t)-np.min(m_t)))))
    if A1_fixed is not None:
        A1 = float(A1_fixed)
    b = float(init.get('b', 0.0))
    s0 = float(init.get('s0', max(1e-3, float(np.std(X[m_t <= np.quantile(m_t, 0.3)])))))
    k = float(init.get('k', 0.3))
    a, c = s0*s0, k*k
    mu = np.clip((m_t-b)/max(A1, 1e-6), 1e-9, float(n_max))
    mu0, lam = float(mu[0]), 0.0
    previous, nll, used = np.inf, np.nan, 0
    for used in range(1, int(n_iter)+1):
        s2 = np.maximum(a + c*N2, 1e-12)
        log_w = np.log(np.maximum(mu, 1e-12))[:, None]*N-mu[:, None]-log_fact[None, :]
        log_w -= _a1cal_lse(log_w, 1)[:, None]
        z = X[:, :, None]-(b+A1*N)
        ll = log_w[:, None, :]-0.5*z*z/s2-0.5*np.log(2.0*np.pi*s2)
        total = _a1cal_lse(ll, 2)
        nll = float(-total.sum())
        r = np.exp(ll-total[:, :, None])
        n_t = np.einsum('tik,k->t', r, N)
        if mode == 'exp':
            mu0, lam = _a1cal_exp_mu(n_t, t_s, n_files)
            mu = np.clip(mu0*np.exp(-lam*t_s), 1e-9, float(n_max))
        else:
            mu = np.clip(n_t/n_files, 1e-9, float(n_max))
        w = r/s2
        Sw = w.sum()
        SwN = np.einsum('tik,k->', w, N)
        SwNN = np.einsum('tik,k->', w, N2)
        wx = w*X[:, :, None]
        Swx = wx.sum()
        SwNx = np.einsum('tik,k->', wx, N)
        if A1_fixed is None:
            det = Sw*SwNN-SwN*SwN
            if abs(det) > 1e-12:
                b = (Swx*SwNN-SwN*SwNx)/det
                A1 = max((Sw*SwNx-SwN*Swx)/det, 1e-4)
        else:
            b = (Swx-A1*SwN)/Sw
        residual = X[:, :, None]-(b+A1*N)
        a, c = _a1cal_var_model(r.sum(axis=(0, 1)),
                                (r*residual*residual).sum(axis=(0, 1)), N2, a, c)
        if abs(previous-nll) < tol*max(1.0, abs(nll)):
            break
        previous = nll
    return {'A1': float(A1), 'b': float(b), 's0': float(np.sqrt(a)), 'k': float(np.sqrt(c)),
            'mu': mu.copy(), 'mu0': float(mu0), 'lam': float(lam), 'nll': float(nll),
            'n_iter': int(used), 'mode': str(mode),
            'T12_ms': float(np.log(2)/lam*1000.0) if lam > 0 else np.nan}


def moment_A1(y):
    """Method of moments: Var(P) = c0 + c1*m + c2*m^2 across time bins."""
    y = np.asarray(y, float)
    m, v = y.mean(axis=0), y.var(axis=0, ddof=1)
    design = np.vstack([np.ones_like(m), m, m*m]).T
    c0, c1, c2 = np.linalg.lstsq(design, v, rcond=None)[0]
    return float(c1/(1.0+c2)), float(c0), float(c1), float(c2), m, v


def _a1cal_simulate(A1, mu0, lam, s0, k, b, n_files, t_s, rng):
    """Poisson injection + independent exponential decay + calibrated noise."""
    n0 = rng.poisson(mu0, int(n_files))
    out = np.empty((int(n_files), len(t_s)), float)
    for i, count in enumerate(n0):
        lifetimes = rng.exponential(1.0/lam, int(count)) if lam > 0 else np.full(int(count), np.inf)
        occupancy = (lifetimes[:, None] > t_s[None, :]).sum(axis=0).astype(float)
        out[i] = b+A1*occupancy+np.sqrt(s0**2+(k*occupancy)**2)*rng.standard_normal(len(t_s))
    return out


# ---------------- Execution ----------------
_a1cal_idx = (np.arange(len(y_files_e)) if A1_CAL_FILE_IDX is None
              else np.asarray(A1_CAL_FILE_IDX, int))
_a1cal_y = np.asarray(y_files_e[_a1cal_idx], float)
_a1cal_t = np.arange(_a1cal_y.shape[1], dtype=float)*float(dt_e)
_a1cal_X = _a1cal_y.T

A1_MOMENT, MOMENT_C0, MOMENT_C1, MOMENT_C2, _a1cal_m, _a1cal_v = moment_A1(_a1cal_y)
_a1cal_init = {'A1': float(np.clip(A1_MOMENT, 0.2, 3.0)), 's0': float(sigma_e), 'k': 0.3}

A1_FIT = fit_power_mixture(_a1cal_X, _a1cal_t, n_max=A1_CAL_NMAX, mode='exp',
                           init=_a1cal_init, n_iter=A1_CAL_EM_ITER)
A1_FIT_FREE = fit_power_mixture(_a1cal_X, _a1cal_t, n_max=A1_CAL_NMAX, mode='free',
                                init=_a1cal_init, n_iter=A1_CAL_EM_ITER)

# Profile likelihood: hold A1 fixed and reoptimize all remaining parameters.
A1_PROFILE_NLL = np.array([
    fit_power_mixture(_a1cal_X, _a1cal_t,
                      n_max=int(max(14, 3.0*np.max(_a1cal_m)/value+5)),
                      mode='exp', A1_fixed=float(value), init=_a1cal_init,
                      n_iter=max(300, A1_CAL_EM_ITER//2))['nll']
    for value in A1_CAL_PROFILE
], float)

# Parametric bootstrap at the fitted point: provides bias AND uncertainty band.
# Time bins within a file are correlated; the likelihood-curvature interval is
# therefore too optimistic and is not used here.
_a1cal_rng = np.random.default_rng(A1_CAL_SEED)
A1_MC = np.array([
    fit_power_mixture(
        _a1cal_simulate(A1_FIT['A1'], A1_FIT['mu0'], A1_FIT['lam'], A1_FIT['s0'],
                        A1_FIT['k'], A1_FIT['b'], len(_a1cal_y), _a1cal_t, _a1cal_rng).T,
        _a1cal_t, n_max=A1_CAL_NMAX, mode='exp', init=_a1cal_init,
        n_iter=max(300, A1_CAL_EM_ITER//2))['A1']
    for _ in range(int(A1_CAL_MC_REPS))
], float)
A1_MC_BIAS = float(np.median(A1_MC)/A1_FIT['A1']-1.0)
A1_CALIBRATED = float(A1_FIT['A1']/(1.0+A1_MC_BIAS))
A1_CAL_SD = float(np.std(A1_MC, ddof=1))
A1_CAL_CI = (float(np.percentile(A1_MC, 16)/(1.0+A1_MC_BIAS)),
             float(np.percentile(A1_MC, 84)/(1.0+A1_MC_BIAS)))

A1_CALIBRATION = {
    'method': 'Poisson-Gauss mixture EM, no staircase fit',
    'A1': A1_CALIBRATED,
    'A1_raw_fit': float(A1_FIT['A1']),
    'A1_moment': float(A1_MOMENT),
    'A1_free_mu': float(A1_FIT_FREE['A1']),
    'sd': A1_CAL_SD,
    'ci68': A1_CAL_CI,
    'mc_bias': A1_MC_BIAS,
    'baseline': float(A1_FIT['b']),
    'sigma0_power': float(A1_FIT['s0']),
    'k_power': float(A1_FIT['k']),
    'mu0': float(A1_FIT['mu0']),
    'T12_ms': float(A1_FIT['T12_ms']),
    'profile_A1': A1_CAL_PROFILE.copy(),
    'profile_nll': A1_PROFILE_NLL.copy(),
    'n_files': int(len(_a1cal_y)),
}

print('='*92)
print('INDEPENDENT A1 CALIBRATION (without staircase fit)')
print(f'Files={len(_a1cal_y)} | time bins={_a1cal_y.shape[1]} | dt={1000*float(dt_e):.3f} ms')
print(f'Method of moments: A1={A1_MOMENT:.3f}   '
      f'(Var = {MOMENT_C0:+.3f} {MOMENT_C1:+.3f}*m {MOMENT_C2:+.4f}*m^2; '
      f'intercept vs sigma^2={float(sigma_e)**2:.3f})')
print(f'Mixture EM      : A1={A1_FIT["A1"]:.3f}  b={A1_FIT["b"]:+.3f}  '
      f'sigma0={A1_FIT["s0"]:.3f}  k={A1_FIT["k"]:.3f}  mu0={A1_FIT["mu0"]:.2f}  '
      f'({A1_FIT["n_iter"]} EM iterations)')
print(f'MC-Bias={A1_MC_BIAS:+.1%} across {A1_CAL_MC_REPS} parametric bootstrap samples')
print(f'-> A1 = {A1_CALIBRATED:.3f} +/- {A1_CAL_SD:.3f}   '
      f'(68%: {A1_CAL_CI[0]:.3f} .. {A1_CAL_CI[1]:.3f})')
print(f'   entered in the notebook: A1_EXTERNAL={A1_EXTERNAL}  -> deviation '
      f'{(float(A1_EXTERNAL)-A1_CALIBRATED)/A1_CAL_SD:+.1f} sigma')
print(f'By-products without staircase fit: T1/2={A1_FIT["T12_ms"]:.1f} ms, '
      f'sigma(N)/A1=sqrt({A1_FIT["s0"]/A1_CALIBRATED:.3f}^2+({A1_FIT["k"]/A1_CALIBRATED:.3f}*N)^2)')
print(f'Robustness check with free mu_t (no decay law): A1={A1_FIT_FREE["A1"]:.3f} '
      f'-- systematically too small; used only as a lower bound.')
if not (A1_CAL_CI[0] <= float(A1_EXTERNAL) <= A1_CAL_CI[1]):
    print('WARNING: A1_EXTERNAL lies outside the 68% calibration band.')
print('A1 applies only to THIS pipeline (NPERSEG, bandwidth, sigma normalization). '
      'If any of these is changed, this cell must be rerun.')
print('='*92)

# ---------------- Diagnostic plots ----------------
_a1cal_fig, _a1cal_ax = plt.subplots(1, 3, figsize=(16.5, 4.9), constrained_layout=True)

_ax = _a1cal_ax[0]
_grid_m = np.linspace(0, _a1cal_m.max()*1.05, 200)
_ax.plot(_a1cal_m, _a1cal_v, 'o', ms=4.5, color='steelblue', alpha=0.7, label='Time bins')
_ax.plot(_grid_m, MOMENT_C0+MOMENT_C1*_grid_m+MOMENT_C2*_grid_m**2, color='black', lw=2,
         label=f'Fit -> A1={A1_MOMENT:.2f}')
_ax.axhline(float(sigma_e)**2, color='tomato', ls=':', lw=1.3, label=r'$\sigma_{bin}^2$')
_ax.set(xlabel='Ensemble mean m(t) [power]', ylabel='Ensemble variance v(t) [power$^2$]',
        title='Method of moments: slope = A1')
_ax.grid(alpha=0.25); _ax.legend(fontsize=8)

_ax = _a1cal_ax[1]
_d2 = 2.0*(A1_PROFILE_NLL-A1_PROFILE_NLL.min())
_ax.plot(A1_CAL_PROFILE, _d2, 'o-', color='#31688e', lw=1.8, ms=5)
_ax.axvspan(A1_CAL_CI[0], A1_CAL_CI[1], color='#35b779', alpha=0.22,
            label='68% (parametric bootstrap)')
_ax.axvline(A1_CALIBRATED, color='#35b779', lw=2.2, label=f'A1={A1_CALIBRATED:.3f}')
_ax.axvline(float(A1_EXTERNAL), color='tomato', ls='--', lw=1.8,
            label=f'A1_EXTERNAL={A1_EXTERNAL}')
_ax.axvline(A1_CALIBRATED/2, color='0.5', ls=':', lw=1.3, label='A1/2 (Alias)')
_ax.set(xlabel='A1 [Power]', ylabel=r'$2\,\Delta$NLL', ylim=(-1, min(60, _d2.max()*1.05)),
        title='Profile likelihood of the mixture')
_ax.grid(alpha=0.25); _ax.legend(fontsize=8)

_ax = _a1cal_ax[2]
_win = np.flatnonzero((A1_FIT['mu'] > 0.25) & (A1_FIT['mu'] < 3.5))
if len(_win) < 3:
    _win = np.arange(min(15, _a1cal_y.shape[1]))
_pool = _a1cal_y[:, _win].ravel()
_hi = float(A1_FIT['b']+A1_CALIBRATED*(np.ceil(A1_FIT['mu'][_win].max())+4))
_pool_view = _pool[(_pool >= -3.0*A1_FIT['s0']) & (_pool <= _hi)]
_edges = np.histogram_bin_edges(_pool_view, bins=55)
_ax.hist(_pool_view, bins=_edges, color='#31688e', alpha=0.75, density=True,
         label=f'Data, time bins {_win[0]}-{_win[-1]}')
_xg = np.linspace(_edges[0], _edges[-1], 600)
_Nv = np.arange(A1_CAL_NMAX+1, dtype=float)
_sN = np.sqrt(A1_FIT['s0']**2+(A1_FIT['k']*_Nv)**2)
_wN = np.exp(np.log(np.maximum(A1_FIT['mu'][_win], 1e-12))[:, None]*_Nv
             - A1_FIT['mu'][_win][:, None]-gammaln(_Nv+1.0)[None, :])
_wN /= _wN.sum(axis=1, keepdims=True)
_wmean = _wN.mean(axis=0)
_comp = (np.exp(-0.5*((_xg[:, None]-(A1_FIT['b']+A1_CALIBRATED*_Nv))/_sN)**2)
         / (_sN*np.sqrt(2*np.pi)))*_wmean
_ax.plot(_xg, _comp.sum(axis=1), color='black', lw=2.0, label='Mixture fit')
for _n in range(min(6, A1_CAL_NMAX+1)):
    if _wmean[_n] > 0.01:
        _ax.plot(_xg, _comp[:, _n], color='tomato', lw=1.0, alpha=0.8)
        _ax.axvline(A1_FIT['b']+A1_CALIBRATED*_n, color='0.4', ls=':', lw=0.8)
_ax.set(xlim=(_edges[0], _edges[-1]), xlabel='Power', ylabel='Density',
        title=f'Mixture in window mu={A1_FIT["mu"][_win].min():.2f}..{A1_FIT["mu"][_win].max():.2f}')
_ax.grid(alpha=0.25); _ax.legend(fontsize=8)
plt.show()


In [ ]:
# === EXACT INTEGER STAIRCASE FIT ===
#
# The signal is reversed in time for the fit. A radioactive decay then becomes
# a strictly monotonic state increase 0 -> 1 -> 2 -> ... . The ion count is part
# of the DP state, making the solution globally optimal within this model. There
# is no subsequent reassignment of levels.

import numpy as np


def exact_integer_staircase(signal, A1, sigma2_by_N, tau2_by_N=None, nmax=None,
                            min_size=1, change_penalty=None):
    """Exact monotonic integer DP with a consistent Gaussian NLL.

    Parameters
    ----------
    signal : 1D array
        Time-reversed power signal.
    A1 : float
        Mean power of one ion.
    sigma2_by_N : 1D array
        Bin-to-bin variance for each integer state N.
    tau2_by_N : 1D array
        Variance of a level offset that is constant across an entire segment.
        This marginalizes the real spread of individual ion amplitudes without
        incorrectly treating it as additional bin noise.
    nmax : int
        Largest allowed integer state.
    min_size : int
        Minimum segment length in bins.
    change_penalty : float
        Penalty per additional breakpoint in units of -2 log L.

    Returns
    -------
    breakpoints, means, lengths, states, score
        As usual, breakpoints include the right edge of the final segment.
    """
    y = np.asarray(signal, float)
    if y.ndim != 1 or len(y) < 2 or np.any(~np.isfinite(y)):
        raise ValueError('signal must be a finite 1D array with at least two bins.')
    if not np.isfinite(A1) or A1 <= 0:
        raise ValueError('A1 must be positive and finite.')

    s2_all = np.asarray(sigma2_by_N, float)
    if s2_all.ndim != 1 or len(s2_all) < 2 or np.any(~np.isfinite(s2_all)) or np.any(s2_all <= 0):
        raise ValueError('sigma2_by_N must be a positive, finite 1D table.')
    if tau2_by_N is None:
        t2_all = np.zeros_like(s2_all)
    else:
        t2_all = np.asarray(tau2_by_N, float)
    if t2_all.shape != s2_all.shape or np.any(~np.isfinite(t2_all)) or np.any(t2_all < 0):
        raise ValueError('tau2_by_N must have the same shape and be nonnegative.')

    n = len(y)
    if nmax is None:
        nmax = max(20, int(np.ceil(max(0.0, np.percentile(y, 99.5))/A1))+4)
    nmax = int(min(max(1, nmax), len(s2_all)-1))
    nlev = nmax+1
    mu = np.arange(nlev, dtype=float)*float(A1)
    s2 = s2_all[:nlev]
    t2 = t2_all[:nlev]
    min_size = int(max(1, min_size))
    if change_penalty is None:
        change_penalty = 2.0*np.log(max(n, 2))
    change_penalty = float(change_penalty)
    if not np.isfinite(change_penalty) or change_penalty < 0:
        raise ValueError('change_penalty must be finite and nonnegative.')

    p1 = np.concatenate(([0.0], np.cumsum(y)))
    p2 = np.concatenate(([0.0], np.cumsum(y*y)))
    inf = float('inf')
    score = np.full((n+1, nlev), inf)
    prev_pos = np.full((n+1, nlev), -1, dtype=np.int16)
    prev_lev = np.full((n+1, nlev), -1, dtype=np.int16)
    strict_value = np.full((n+1, nlev), inf)
    strict_arg = np.full((n+1, nlev), -1, dtype=np.int16)

    def segment_cost(starts, end):
        starts = np.asarray(starts, int)
        length = (end-starts).astype(float)
        total = p1[end]-p1[starts]
        square = p2[end]-p2[starts]
        mean = total/length
        rss = np.maximum(square-total*total/length, 0.0)
        den = s2[None, :]+length[:, None]*t2[None, :]
        # The omitted term n*log(2*pi) is identical for all paths.
        return (rss[:, None]/s2[None, :]
                +(length[:, None]-1.0)*np.log(s2[None, :])
                +np.log(den)
                +length[:, None]*(mean[:, None]-mu[None, :])**2/den)

    for end in range(min_size, n+1):
        base = segment_cost(np.array([0]), end)[0]
        score[end] = base
        prev_pos[end] = 0

        starts = np.arange(min_size, end-min_size+1, dtype=int)
        if len(starts):
            candidate = strict_value[starts]+change_penalty+segment_cost(starts, end)
            jj = np.argmin(candidate, axis=0)
            best = candidate[jj, np.arange(nlev)]
            take = best < score[end]
            if np.any(take):
                chosen_start = starts[jj]
                score[end, take] = best[take]
                prev_pos[end, take] = chosen_start[take]
                prev_lev[end, take] = strict_arg[chosen_start[take], np.flatnonzero(take)]

        best_value = inf
        best_level = -1
        for level in range(nlev):
            strict_value[end, level] = best_value
            strict_arg[end, level] = best_level
            if score[end, level] < best_value:
                best_value = score[end, level]
                best_level = level

    last = int(np.argmin(score[n]))
    if not np.isfinite(score[n, last]):
        raise RuntimeError('No admissible integer staircase path found.')

    segments = []
    end, level = n, last
    while True:
        start = int(prev_pos[end, level])
        if start < 0:
            raise RuntimeError('Invalid backtracking in the integer staircase fit.')
        segments.append((start, end, level))
        if start == 0:
            break
        old_level = int(prev_lev[end, level])
        if old_level < 0 or old_level >= level:
            raise RuntimeError('Nonmonotonic backtracking in the integer staircase fit.')
        end, level = start, old_level
    segments.reverse()

    breakpoints = [int(end) for _, end, _ in segments]
    lengths = np.asarray([end-start for start, end, _ in segments], int)
    states = np.asarray([level for _, _, level in segments], int)
    means = np.asarray([(p1[end]-p1[start])/(end-start)
                        for start, end, _ in segments], float)
    if breakpoints[-1] != n or lengths.sum() != n or np.any(np.diff(states) <= 0):
        raise RuntimeError('Internal invariant of the integer staircase fit violated.')
    return breakpoints, means, lengths, states, float(score[n, last])



In [ ]:
# === REAL STAIRCASE FIT: physically regularized integer model ===
#
# A1 comes from an independent high-resolution Ge calibration. The latent
# model staircase is exactly N*A1 and strictly monotonic. At this stage, bin
# noise is described by the ground-state width as before. The following cell
# measures its actual dependence on N for use in the simulation generator. Real
# and simulated data continue to pass through exactly the same staircase fit.

import numpy as np
import matplotlib.pyplot as plt

# ---------------- Parameters for this data set ----------------
FILE_START = 0
BAD_RANGES = ()
EXACT_DP_PENALTY_FACTOR = 0.30  # penalty = factor * 2*log(number of time bins)
EXACT_DP_MIN_SIZE = 1           # short physical levels remain allowed
EXACT_DP_NMAX_CAP = 80
PLOT_EVERY = 10
PLOT_FILES = {0, 15}
# -----------------------------------------------------------------


def is_bad_file(file_index):
    return any(lo <= int(file_index) <= hi for lo, hi in BAD_RANGES)


FIT_FILE_IDX = np.asarray([
    i for i in range(FILE_START, len(y_files_e)) if not is_bad_file(i)
], int)
if len(FIT_FILE_IDX) == 0:
    raise RuntimeError('No file remains for the fit after FILE_START/BAD_RANGES.')

# A1 source: True uses the value measured from the data in the calibration
# cell; False uses the fixed A1_EXTERNAL value.
USE_CALIBRATED_A1 = True

if USE_CALIBRATED_A1:
    A1 = float(A1_CALIBRATED)
    A1_SOURCE = f'calibration cell (+/- {A1_CAL_SD:.3f})'
else:
    if A1_EXTERNAL is None:
        raise ValueError(
            'A1_EXTERNAL is None. Either set USE_CALIBRATED_A1=True or '
            'enter the mean single-ion power in cell 2.'
        )
    A1 = float(A1_EXTERNAL)
    A1_SOURCE = 'A1_EXTERNAL (fixed value)'
if not np.isfinite(A1) or A1 <= 0:
    raise ValueError('A1_EXTERNAL must be positive and finite.')

sigP = float(sigma_e)
if not np.isfinite(sigP) or sigP <= 0:
    raise ValueError('sigma_e must be positive and finite.')

_ground_tail_bins = min(15, y_files_e.shape[1])
_ground_means = np.asarray(
    y_files_e[np.asarray(GROUND_NOISE_IDX, int), -_ground_tail_bins:].mean(axis=1),
    float,
)
EXACT_GROUND_LEVEL = float(np.median(_ground_means))
_ground_mad = 1.4826*float(np.median(np.abs(_ground_means-EXACT_GROUND_LEVEL)))
_ground_measurement_sd = sigP/np.sqrt(_ground_tail_bins)
EXACT_BASELINE_SD = float(np.sqrt(max(
    _ground_mad**2-_ground_measurement_sd**2,
    (0.02*A1)**2,
)))

_state_grid = np.arange(EXACT_DP_NMAX_CAP+1, dtype=float)
EXACT_SIGMA2_BY_N = np.full(EXACT_DP_NMAX_CAP+1, sigP**2, float)

EXACT_TAU2_BY_N = np.full(
    EXACT_DP_NMAX_CAP+1, EXACT_BASELINE_SD**2, float
)
EXACT_CHANGE_PENALTY = float(
    EXACT_DP_PENALTY_FACTOR*2.0*np.log(y_files_e.shape[1])
)


def fit_integer_dataset(data, file_ids=None):
    """Applies exactly the same integer DP model to real or simulated data."""
    data = np.asarray(data, float)
    if data.ndim != 2 or data.shape[1] < 2 or np.any(~np.isfinite(data)):
        raise ValueError('data must be a finite 2D array (files x time bins).')
    if file_ids is None:
        file_ids = np.arange(len(data), dtype=int)
    file_ids = np.asarray(file_ids, int)
    if len(file_ids) != len(data):
        raise ValueError('file_ids and data have different lengths.')

    medians, lengths, states, scores = [], [], [], []
    for row in data:
        reversed_row = np.asarray(row[::-1], float)
        nmax = max(20, int(np.ceil(
            max(0.0, np.percentile(reversed_row, 99.5))/A1
        ))+4)
        nmax = min(EXACT_DP_NMAX_CAP, nmax)
        _, mean, length, state, score = exact_integer_staircase(
            reversed_row,
            A1,
            EXACT_SIGMA2_BY_N,
            EXACT_TAU2_BY_N,
            nmax=nmax,
            min_size=EXACT_DP_MIN_SIZE,
            change_penalty=EXACT_CHANGE_PENALTY,
        )
        medians.append(np.asarray(mean, float))
        lengths.append(np.asarray(length, int))
        states.append(np.asarray(state, int))
        scores.append(float(score))
    return medians, lengths, states, np.asarray(scores, float)


med_m, len_m, ion_m, EXACT_DP_SCORES = fit_integer_dataset(
    y_files_e[FIT_FILE_IDX], FIT_FILE_IDX
)
idx_list = FIT_FILE_IDX.copy()

# med_m contains only the observed segment means (diagnostic). The actual
# latent fit staircase of the integer model is N*A1. It must increase strictly in
# reversed time and therefore decrease strictly in forward time.
REAL_FIT_LEVELS = [A1*np.asarray(state, float) for state in ion_m]
if any(np.any(np.diff(level) <= 0) for level in REAL_FIT_LEVELS):
    raise RuntimeError('Internal error: the real model staircase is not strictly monotonic.')
REAL_RAW_MEAN_REVERSALS = int(sum(
    np.count_nonzero(np.diff(np.asarray(mean, float)) < 0) for mean in med_m
))

print('='*88)
print('REAL EXACT INTEGER STAIRCASE FIT')
print(f'Files={len(FIT_FILE_IDX)} | A1={A1:.4f} [{A1_SOURCE}] | sigma_bin={sigP:.4f}')
print(f'Baseline-SD={EXACT_BASELINE_SD:.4f} ({EXACT_BASELINE_SD/A1:.3f} A1) '
      f'| breakpoint penalty={EXACT_CHANGE_PENALTY:.3f} '
      f'({EXACT_DP_PENALTY_FACTOR:.2f}*2log(n))')
print(f'{sum(len(v) for v in ion_m)} segments '
      f'({np.mean([len(v) for v in ion_m]):.2f} per file)')
print(f'Monotonicity: model staircase 0 reversals | '
      f'observed segment means {REAL_RAW_MEAN_REVERSALS} reversals')

# Direct check that the observed segment means lie around N*A1.
_state_values = {}
for _mean, _state in zip(med_m, ion_m):
    for _value, _n in zip(_mean, _state):
        _state_values.setdefault(int(_n), []).append(float(_value/A1))
EXACT_LEVEL_MEDIANS_OVER_A1 = {
    n: float(np.median(v)) for n, v in _state_values.items() if len(v) >= 3
}
print('Median of the observed segment means in A1 units:')
print('  '+'  '.join(
    f'N{n}:{EXACT_LEVEL_MEDIANS_OVER_A1[n]:.3f}'
    for n in sorted(EXACT_LEVEL_MEDIANS_OVER_A1) if n <= 10
))
_low_deviation = [
    abs(value-n) for n, value in EXACT_LEVEL_MEDIANS_OVER_A1.items() if n <= 6
]
if _low_deviation and max(_low_deviation) > 0.35:
    print('WARNING: At least one low level deviates by >0.35 A1 from the integer value; '
          'inspect A1/bands visually.')
print('='*88)

# Diagnostic plots: red is the strictly monotonic latent model staircase N*A1.
# The raw segment mean is shown only as a number for noise/amplitude diagnostics.
for _mean, _fit, _length, _state, _file in zip(
        med_m, REAL_FIT_LEVELS, len_m, ion_m, idx_list):
    if int(_file) not in PLOT_FILES and (int(_file)-FILE_START) % PLOT_EVERY != 0:
        continue
    _edges = np.concatenate([[0], np.cumsum(_length)]).astype(int)
    _y = np.asarray(y_files_e[int(_file)][::-1], float)/A1
    fig, ax = plt.subplots(figsize=(12, 4.6), constrained_layout=True)
    ax.plot(np.arange(len(_y)), _y, color='steelblue', alpha=0.55, lw=0.9,
            label='Data (time reversed)')
    for _n in range(int(max(_state.max(), np.ceil(np.nanmax(_y))))+1):
        ax.axhline(_n, color='0.5', ls=':', lw=0.45, alpha=0.35)
    for _j, (_start, _end) in enumerate(zip(_edges[:-1], _edges[1:])):
        _fit_level = _fit[_j]/A1
        _raw_mean = _mean[_j]/A1
        ax.hlines(_fit_level, _start, _end, color='tomato', lw=2.6, zorder=3,
                  label='monotonic model staircase N*A1' if _j == 0 else None)
        if _j:
            ax.vlines(_start, _fit[_j-1]/A1, _fit_level,
                      color='tomato', lw=1.3, zorder=3)
        ax.text((_start+_end)/2, _fit_level+0.18,
                f'N={_state[_j]} | mean={_raw_mean:.2f}',
                ha='center', va='bottom', fontsize=8,
                color=('green' if _state[_j] == 0 else 'purple'))
    ax.set(xlabel='Time bin (reversed: left = late/ground state)',
           ylabel='Power / A1',
           title=f'File {int(_file)}: exact integer staircase fit')
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8)
    plt.show()

# Histogram of observed segment means: each segment contributes one count.
# These measured means may fluctuate; they are not the red latent fit staircase.
REAL_LEVEL_HEIGHTS = np.concatenate([np.asarray(v, float) for v in med_m])
REAL_LEVEL_HEIGHTS = REAL_LEVEL_HEIGHTS[np.isfinite(REAL_LEVEL_HEIGHTS)]
LEVEL_HIST_BIN_WIDTH_A1 = 0.05
_hist_width = LEVEL_HIST_BIN_WIDTH_A1*A1
_hist_low = min(-0.5*A1, np.floor(np.quantile(REAL_LEVEL_HEIGHTS, 0.001)/_hist_width)*_hist_width)
_hist_high = max(10.0*A1, np.ceil(np.quantile(REAL_LEVEL_HEIGHTS, 0.995)/_hist_width)*_hist_width)
REAL_LEVEL_HIST_EDGES = np.arange(_hist_low, _hist_high+1.001*_hist_width, _hist_width)

fig, ax = plt.subplots(figsize=(11.5, 5.0), constrained_layout=True)
ax.hist(REAL_LEVEL_HEIGHTS, bins=REAL_LEVEL_HIST_EDGES,
        color='#31688e', alpha=0.84, edgecolor='none')
for _n in range(max(0, int(np.floor(_hist_high/A1)))+1):
    ax.axvline(_n*A1, color='black', lw=0.55, alpha=0.22)
ax.set(xlabel='Observed segment mean / power',
       ylabel='Counts (one count per segment)',
       title='Real data: observed segment means from the staircase fit',
       xlim=(REAL_LEVEL_HIST_EDGES[0], REAL_LEVEL_HIST_EDGES[-1]))
ax.grid(axis='y', alpha=0.22)
plt.show()
print(f'Real histogram: {len(REAL_LEVEL_HEIGHTS)} levels '
      f'| bin width={LEVEL_HIST_BIN_WIDTH_A1:.2f} A1')



In [ ]:
# === NOISE MODEL sigma(N): CALIBRATED FROM REAL PLATEAUS ===
#
# The mean plateau height remains exactly N*A1. Separately, this cell measures
# the bin-to-bin spread within sufficiently long real plateaus. As in run, the
# dependence is described by
#
#     sigma(N)/A1 = sqrt(sigma0**2 + (k*N)**2)
#
# Because of the substantially shorter lifetime, 10-bin plateaus are too rare
# here; 5 bins are a documented compromise. The c4 correction removes the known
# downward bias of the sample standard deviation for short segments. For this
# small data set, the smooth fit curve is more stable than interpolation between
# sparsely populated high-N support points.

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import gammaln

# ---------------- Noise-calibration parameters ----------------
NOISE_MINLEN = 5
NOISE_MAX_N = 6
# With 102 files this gives 10 segments; for smaller data sets the threshold
# is reduced to a minimum of 4 segments and is always reported in the output.
NOISE_MIN_SEGMENTS = min(
    10, max(4, int(np.ceil(0.10*len(FIT_FILE_IDX))))
)
# ------------------------------------------------------------------


def _c4_std_bias(n):
    """E[s]/sigma for the sample SD of a normal distribution."""
    n = float(n)
    if n < 2:
        return np.nan
    return float(
        np.sqrt(2.0/(n-1.0))
        * np.exp(gammaln(n/2.0)-gammaln((n-1.0)/2.0))
    )


_noise_by_N = {}
_noise_level_all = []
_noise_sigma_all = []
_noise_state_all = []
for _file, _mean, _length, _state in zip(idx_list, med_m, len_m, ion_m):
    _y_rev_A1 = np.asarray(y_files_e[int(_file)][::-1], float)/A1
    _edges = np.concatenate([[0], np.cumsum(_length)]).astype(int)
    for _j, (_start, _end, _n) in enumerate(
            zip(_edges[:-1], _edges[1:], np.asarray(_state, int))):
        _m = int(_end-_start)
        if _m < NOISE_MINLEN:
            continue
        _segment = _y_rev_A1[_start:_end]
        _c4 = _c4_std_bias(_m)
        _sigma_corr = float(np.std(_segment, ddof=1)/_c4)
        if not np.isfinite(_sigma_corr) or _sigma_corr <= 0:
            continue
        _noise_by_N.setdefault(int(_n), []).append(_sigma_corr)
        _noise_level_all.append(float(np.mean(_segment)))
        _noise_sigma_all.append(_sigma_corr)
        _noise_state_all.append(int(_n))

NOISE_N = np.asarray(sorted(
    n for n, values in _noise_by_N.items()
    if n <= NOISE_MAX_N and len(values) >= NOISE_MIN_SEGMENTS
), int)
NOISE_SIGMA_A1 = np.asarray([
    np.median(_noise_by_N[int(n)]) for n in NOISE_N
], float)
NOISE_COUNTS = np.asarray([
    len(_noise_by_N[int(n)]) for n in NOISE_N
], int)

if len(NOISE_N) < 2 or not np.any(NOISE_N > 0):
    raise RuntimeError(
        'Too few populated ion counts for sigma(N). '
        'Check NOISE_MINLEN/NOISE_MIN_SEGMENTS; the ground state and '
        'at least one N>0 level are required.'
    )


def _sigma_model(n, sigma0, k):
    n = np.asarray(n, float)
    return np.sqrt(sigma0*sigma0+(k*n)**2)


_i0 = np.flatnonzero(NOISE_N == 0)
_sigma0_guess = (
    float(NOISE_SIGMA_A1[_i0[0]]) if len(_i0)
    else float(np.min(NOISE_SIGMA_A1))
)
_positive = NOISE_N > 0
_k_candidates = np.sqrt(np.maximum(
    NOISE_SIGMA_A1[_positive]**2-_sigma0_guess**2, 0.0
))/NOISE_N[_positive]
_k_guess = max(0.05, float(np.median(_k_candidates)))

if len(NOISE_N) >= 3:
    # sigma=1/sqrt(count) corresponds to sqrt(count) weighting of the robust
    # median support points; absolute_sigma=False because these are not measurement-error bars.
    _noise_popt, NOISE_FIT_COV = curve_fit(
        _sigma_model,
        NOISE_N.astype(float),
        NOISE_SIGMA_A1,
        p0=(_sigma0_guess, _k_guess),
        sigma=1.0/np.sqrt(NOISE_COUNTS.astype(float)),
        absolute_sigma=False,
        bounds=([1e-6, 0.0], [np.inf, np.inf]),
        maxfev=20000,
    )
    SIGMA0_FIT, K_FIT = map(float, _noise_popt)
else:
    # Small-data-set fallback for exactly two support points: the ground state sets
    # sigma0 and the positive level sets k. The limited robustness is clearly reported.
    SIGMA0_FIT = _sigma0_guess
    K_FIT = _k_guess
    NOISE_FIT_COV = np.full((2, 2), np.nan)


def SIGMA_OF_N(n):
    """Fitted bin SD in A1 units at occupancy n (including fractional occupancy)."""
    n = np.maximum(np.asarray(n, float), 0.0)
    return _sigma_model(n, SIGMA0_FIT, K_FIT)


NOISE_SIGMA0_POWER = float(A1*SIGMA0_FIT)
NOISE_K_POWER = float(A1*K_FIT)
NOISE_CALIBRATION = {
    'min_length_bins': int(NOISE_MINLEN),
    'min_segments_per_state': int(NOISE_MIN_SEGMENTS),
    'states': NOISE_N.copy(),
    'sigma_A1': NOISE_SIGMA_A1.copy(),
    'counts': NOISE_COUNTS.copy(),
    'sigma0_A1': float(SIGMA0_FIT),
    'k_A1': float(K_FIT),
}

print('='*88)
print('NOISE CALIBRATION FROM REAL STAIRCASE PLATEAUS')
print(f'Minimum length={NOISE_MINLEN} Bins | c4-corrected SD '
      f'| minimum count/support point={NOISE_MIN_SEGMENTS}')
for _n, _sigma, _count in zip(NOISE_N, NOISE_SIGMA_A1, NOISE_COUNTS):
    print(f'  N={int(_n)}: median sigma={_sigma:.3f} A1 '
          f'({A1*_sigma:.4f} Power; n_seg={int(_count)})')
print(f'Fit sigma(N)/A1=sqrt(sigma0^2+(k*N)^2): '
      f'sigma0={SIGMA0_FIT:.3f}, k={K_FIT:.3f}')
print(f'In power units: sigma0={NOISE_SIGMA0_POWER:.4f}, '
      f'k={NOISE_K_POWER:.4f}')
if len(NOISE_N) == 2:
    print('WARNING: Only two support points; sigma(N) is correspondingly uncertain.')
print('The curve controls the simulation generator. Real and simulated data '
      'continue to be reconstructed with the same unchanged staircase fit.')
print('='*88)

_noise_level_all = np.asarray(_noise_level_all, float)
_noise_sigma_all = np.asarray(_noise_sigma_all, float)
_xmax = max(
    float(NOISE_MAX_N+1),
    min(12.0, float(np.ceil(np.quantile(_noise_level_all, 0.98))))
)
_xf = np.linspace(0.0, _xmax, 300)
fig, ax = plt.subplots(figsize=(10.5, 6.0), constrained_layout=True)
ax.scatter(
    _noise_level_all, _noise_sigma_all,
    s=12, color='steelblue', alpha=0.28,
    label=f'individual plateaus (length >= {NOISE_MINLEN} Bins)',
)
ax.plot(
    NOISE_N, NOISE_SIGMA_A1, 'o', color='darkred', ms=7,
    label='Median for each ion count used', zorder=4,
)
for _n, _sigma, _count in zip(NOISE_N, NOISE_SIGMA_A1, NOISE_COUNTS):
    ax.annotate(
        f'n={int(_count)}', (_n, _sigma), xytext=(5, 5),
        textcoords='offset points', fontsize=8, color='darkred',
    )
ax.plot(
    _xf, SIGMA_OF_N(_xf), color='black', lw=2.0,
    label=(r'Fit: $\sigma(N)=\sqrt{\sigma_0^2+(kN)^2}$ '
           f'({SIGMA0_FIT:.2f}, {K_FIT:.2f})'),
)
ax.set(
    xlabel='Observed plateau mean / A1',
    ylabel='SD within the plateau / A1',
    title='Real data: bin noise versus level height',
    xlim=(-0.5, _xmax),
)
ax.set_ylim(bottom=0.0)
ax.grid(alpha=0.25)
ax.legend()
plt.show()



In [ ]:
# === T1/2 HEATMAP: discrete maximum likelihood  ===


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

DISC_L_VALUES = np.asarray([5, 8, 10, 12, 15, 17, 20, 25], int)
DISC_G_VALUES = np.asarray([5, 8, 10, 12, 15, 20, 25], int)
DISC_RATE_BOUNDS = (0.1, 100.0)  # decay rate [1/s]
DISC_MIN_EVENTS = 8
DISC_MIN_INFORMATIVE = 5
DISC_BOOTSTRAP = 250
DISC_BLOCK_FILES = min(8, max(1, len(FIT_FILE_IDX)//4))
DISC_SEED = 20260827
_disc_dt = float(dt_e)


def extract_terminal_10_events(length_paths, state_paths, file_ids, n_time):
    """Extracts at most the terminal, resolved 1->0 level from each file."""
    rows = []
    for length, state, file_id in zip(length_paths, state_paths, file_ids):
        length = np.asarray(length, int)
        state = np.asarray(state, int)
        if len(length) != len(state) or len(length) == 0 or np.any(length < 1):
            raise RuntimeError(f'Invalid staircase path in file {int(file_id)}.')
        if int(length.sum()) != int(n_time):
            raise RuntimeError(f'Paths in file {int(file_id)} do not cover the time window.')
        if len(state) > 1 and np.any(np.diff(state) <= 0):
            raise RuntimeError(f'Non-strictly-monotonic integer path in file {int(file_id)}.')
        if len(state) >= 2 and state[0] == 0 and state[1] == 1:
            g = int(length[0])
            k = int(length[1])
            rows.append((int(file_id), k, g, k+g))
    dtype = [('file', 'i8'), ('k', 'i8'), ('g', 'i8'), ('H', 'i8')]
    return np.asarray(rows, dtype=dtype)


def fit_discrete_1ion(k, lo, hi):
    """MLE of the doubly truncated geometric distribution in integer bins."""
    k = np.asarray(k, int)
    lo = np.broadcast_to(np.asarray(lo, int), k.shape)
    hi = np.asarray(hi, int)
    if k.shape != hi.shape:
        raise ValueError('k and hi must have the same shape.')
    if len(k) == 0:
        return np.nan, 0, np.nan, False, 0
    if np.any(lo < 1) or np.any(hi < lo) or np.any(k < lo) or np.any(k > hi):
        raise ValueError('Invalid discrete truncation interval.')

    width = hi-lo+1
    informative = int(np.count_nonzero(width > 1))
    n_events = int(len(k))
    if n_events < DISC_MIN_EVENTS or informative < DISC_MIN_INFORMATIVE:
        return np.nan, n_events, np.nan, False, informative
    distance = k-lo

    def nll(rate):
        eta = float(rate)*_disc_dt
        log_event = np.log(-np.expm1(-eta))
        log_norm = np.log(-np.expm1(-eta*width))
        return float(-(n_events*log_event-eta*np.sum(distance)-np.sum(log_norm)))

    result = minimize_scalar(
        nll,
        bounds=tuple(map(float, DISC_RATE_BOUNDS)),
        method='bounded',
        options={'xatol': 1e-10, 'maxiter': 500},
    )
    if not result.success or not np.isfinite(result.fun):
        return np.nan, n_events, np.nan, False, informative
    rate = float(result.x)
    span = DISC_RATE_BOUNDS[1]-DISC_RATE_BOUNDS[0]
    boundary = bool(
        rate-DISC_RATE_BOUNDS[0] < 1e-5*span
        or DISC_RATE_BOUNDS[1]-rate < 1e-5*span
    )
    return float(np.log(2)/rate*1000.0), n_events, rate, boundary, informative


def discrete_matrix(events):
    shape = (len(DISC_G_VALUES), len(DISC_L_VALUES))
    half_life = np.full(shape, np.nan)
    counts = np.zeros(shape, int)
    rates = np.full(shape, np.nan)
    boundary = np.zeros(shape, bool)
    informative = np.zeros(shape, int)
    for i_g, G in enumerate(DISC_G_VALUES):
        hi = events['H']-int(G)
        for i_l, L in enumerate(DISC_L_VALUES):
            lo = int(L)+1
            use = (events['g'] >= G) & (events['k'] >= lo)
            result = fit_discrete_1ion(events['k'][use], lo, hi[use])
            (half_life[i_g, i_l], counts[i_g, i_l], rates[i_g, i_l],
             boundary[i_g, i_l], informative[i_g, i_l]) = result
    return half_life, counts, rates, boundary, informative


def bootstrap_discrete_matrix(events, all_file_ids, draws, block_files, seed):
    """Circular moving-block bootstrap over files, not only over events."""
    all_file_ids = np.asarray(all_file_ids, int)
    if len(all_file_ids) == 0 or len(np.unique(all_file_ids)) != len(all_file_ids):
        raise ValueError('all_file_ids must be nonempty and unique.')
    block_files = int(min(max(1, block_files), max(1, len(all_file_ids)//2)))
    event_at_position = np.full(len(all_file_ids), -1, int)
    lookup = {int(file_id): pos for pos, file_id in enumerate(all_file_ids)}
    for event_index, file_id in enumerate(events['file']):
        if int(file_id) not in lookup:
            raise RuntimeError('An event file is missing from all_file_ids.')
        event_at_position[lookup[int(file_id)]] = event_index

    rng = np.random.default_rng(seed)
    boot = np.full(
        (int(draws), len(DISC_G_VALUES), len(DISC_L_VALUES)), np.nan
    )
    n_files = len(all_file_ids)
    for b in range(int(draws)):
        positions = []
        while len(positions) < n_files:
            start = int(rng.integers(0, n_files))
            positions.extend(((start+np.arange(block_files)) % n_files).tolist())
        event_indices = event_at_position[np.asarray(positions[:n_files], int)]
        event_indices = event_indices[event_indices >= 0]
        boot[b] = discrete_matrix(events[event_indices])[0]
    return boot, block_files


def bootstrap_summary(boot):
    sd = np.full(boot.shape[1:], np.nan)
    ci95 = np.full((2,)+boot.shape[1:], np.nan)
    valid = np.zeros(boot.shape[1:], int)
    for i_g in range(boot.shape[1]):
        for i_l in range(boot.shape[2]):
            values = boot[:, i_g, i_l]
            values = values[np.isfinite(values)]
            valid[i_g, i_l] = len(values)
            if len(values) >= 2:
                sd[i_g, i_l] = float(np.std(values, ddof=1))
                ci95[:, i_g, i_l] = np.percentile(values, [2.5, 97.5])
    return sd, ci95, valid


def draw_discrete_heatmap(ax, half_life, error, counts, title, vmin=None, vmax=None):
    image = ax.imshow(
        half_life,
        origin='lower',
        aspect='auto',
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
        extent=[-.5, len(DISC_L_VALUES)-.5, -.5, len(DISC_G_VALUES)-.5],
    )
    ax.set_xticks(range(len(DISC_L_VALUES)), DISC_L_VALUES)
    ax.set_yticks(range(len(DISC_G_VALUES)), DISC_G_VALUES)
    ax.set_xlabel(r'strict cut $k>L$  ($L$ [Bins])')
    ax.set_ylabel(r'minimum terminal ground state $g\geq G$  ($G$ [Bins])')
    ax.set_title(title)
    finite = half_life[np.isfinite(half_life)]
    middle = float(np.nanmedian(finite)) if len(finite) else np.nan
    for i_g in range(half_life.shape[0]):
        for i_l in range(half_life.shape[1]):
            value = half_life[i_g, i_l]
            if np.isfinite(value):
                err = error[i_g, i_l]
                err_text = f'{err:.0f}' if np.isfinite(err) else '?'
                ax.text(
                    i_l, i_g, f'{value:.0f}\n+/-{err_text} ({counts[i_g, i_l]})',
                    ha='center', va='center', fontsize=6.3,
                    color='white' if value <= middle else 'black',
                )
    return image


def print_discrete_matrix(label, half_life, error, counts):
    print('\n'+label)
    print('  G\\L '+' '.join(f'{L:>18d}' for L in DISC_L_VALUES))
    for i_g, G in enumerate(DISC_G_VALUES):
        fields = []
        for i_l in range(len(DISC_L_VALUES)):
            value = half_life[i_g, i_l]
            if np.isfinite(value):
                err = error[i_g, i_l]
                fields.append(
                    f'{value:6.1f}+/-{err:4.1f}({counts[i_g, i_l]:4d})'
                    if np.isfinite(err) else
                    f'{value:6.1f}+/- nan({counts[i_g, i_l]:4d})'
                )
            else:
                fields.append(f'{"nan":>18s}')
        print(f'  {G:3d} '+' '.join(fields))


# Small formula test with known geometric random numbers and a variable horizon.
_toy_rng = np.random.default_rng(271828)
_toy_truth_ms = 37.0
_toy_rate = np.log(2)/(_toy_truth_ms/1000.0)
_toy_k = _toy_rng.geometric(1-np.exp(-_toy_rate*_disc_dt), size=120000)
_toy_H = _toy_rng.integers(35, 101, size=len(_toy_k))
_toy_lo, _toy_G = 11, 15
_toy_hi = _toy_H-_toy_G
_toy_use = (_toy_k >= _toy_lo) & (_toy_k <= _toy_hi)
DISC_TOY_T12_MS = fit_discrete_1ion(
    _toy_k[_toy_use], _toy_lo, _toy_hi[_toy_use]
)[0]
if not np.isfinite(DISC_TOY_T12_MS) or abs(DISC_TOY_T12_MS-_toy_truth_ms) > 1.0:
    raise AssertionError('The discrete likelihood fails the internal toy test.')

REAL_EVENTS = extract_terminal_10_events(
    len_m, ion_m, FIT_FILE_IDX, y_files_e.shape[1]
)
if len(REAL_EVENTS) == 0:
    raise RuntimeError('No terminal 1->0 event found in the real staircase fit.')
T_real, N_real, RATE_real, BOUND_real, INFO_real = discrete_matrix(REAL_EVENTS)
BOOT_real, DISC_BLOCK_FILES_USED = bootstrap_discrete_matrix(
    REAL_EVENTS,
    FIT_FILE_IDX,
    DISC_BOOTSTRAP,
    DISC_BLOCK_FILES,
    DISC_SEED,
)
E_real, CI_real, NB_real = bootstrap_summary(BOOT_real)

print('='*96)
print('REAL DISCRETE T1/2 ANALYSIS -- NO HALF-BIN')
print(f'dt={1000*_disc_dt:.3f} ms | terminal 1->0 events={len(REAL_EVENTS)} '
      f'| Bootstrap={DISC_BOOTSTRAP} x {DISC_BLOCK_FILES_USED} Files')
print(f'Toy test: {_toy_truth_ms:.1f} ms -> {DISC_TOY_T12_MS:.2f} ms')
print_discrete_matrix('T1/2 +/- block-bootstrap SD [ms] (event count)',
                      T_real, E_real, N_real)
if np.any(BOUND_real):
    print('WARNING: At least one MLE lies at DISC_RATE_BOUNDS.')
print('\nCRITICAL INTERPRETATION: L corrects only a hard detection threshold that is '
      'fully efficient above L. A stable T1/2 plateau versus L remains the '
      'decisive check; small event counts are reported in every cell.')
print('='*96)

real_discrete_results = {
    'model': 'event-conditioned doubly truncated geometric likelihood',
    'dt_s': _disc_dt,
    'L': DISC_L_VALUES.copy(),
    'G': DISC_G_VALUES.copy(),
    'T12_ms': T_real,
    'bootstrap_sd_ms': E_real,
    'bootstrap_ci95_ms': CI_real,
    'n_events': N_real,
    'events': REAL_EVENTS.copy(),
    'bootstrap_block_files': int(DISC_BLOCK_FILES_USED),
    'toy_T12_ms': float(DISC_TOY_T12_MS),
}


In [ ]:
# === PURE DECAY SIMULATION + IDENTICAL STAIRCASE FIT ===
#
# The simulation starts at the center of the first analyzed STFT bin. A
# Poisson-distributed initial ion count is drawn for each file, and each ion is
# assigned an exponential lifetime. The noiseless plateau height is fixed at
# N*A1: there is no additional spread of individual ion amplitudes and no random
# file baseline offset. Only a decay within the STFT window briefly produces a
# fractional effective occupancy N_eff through the Hann^2 window kernel. Around
# N_eff*A1, bin noise sigma(N_eff), calibrated from real plateaus in the preceding
# cell, is added. Stable contamination is not simulated.

import numpy as np
from scipy.signal import get_window

# ---------------- Simulation parameters (CHANGE HERE) ----------------
SIM_MASTER_SEED = 20260828
SIM_T12_TRUE_MS = 37.0
SIM_NFILES = 1000  # for a more stable simulation heatmap; freely adjustable here
# None matches the mean to the real power in the first analyzed bin.
SIM_INITIAL_MEAN_IONS = None
# -----------------------------------------------------------------------

SIM_A1_TRUE = float(A1)
SIM_DT_S = float(dt_e)
SIM_NT = int(y_files_e.shape[1])
SIM_TIME_S = np.arange(SIM_NT, dtype=float)*SIM_DT_S
SIM_WINDOW_S = float(NPERSEG)/float(fs)
SIM_RATE_TRUE_S = float(np.log(2)/(SIM_T12_TRUE_MS/1000.0))

if SIM_NFILES < 1:
    raise ValueError('SIM_NFILES must be at least 1.')
_real_first_bin_mean_ions = float(np.mean(y_files_e[FIT_FILE_IDX, 0])/A1)
if SIM_INITIAL_MEAN_IONS is None:
    SIM_INITIAL_MEAN_IONS_USED = max(0.05, _real_first_bin_mean_ions)
else:
    SIM_INITIAL_MEAN_IONS_USED = float(SIM_INITIAL_MEAN_IONS)
if not np.isfinite(SIM_INITIAL_MEAN_IONS_USED) or SIM_INITIAL_MEAN_IONS_USED <= 0:
    raise ValueError('SIM_INITIAL_MEAN_IONS must be positive and finite or None.')

_seed_sequence = np.random.SeedSequence(SIM_MASTER_SEED)
_rng_n0, _rng_tau, _rng_noise = [
    np.random.default_rng(seed) for seed in _seed_sequence.spawn(3)
]

_noise_tail_bins = min(15, y_files_e.shape[1])
_noise_blocks = np.asarray(
    y_files_e[np.asarray(GROUND_NOISE_IDX, int), -_noise_tail_bins:], float
)
_noise_blocks = _noise_blocks-_noise_blocks.mean(axis=1, keepdims=True)
_noise_scale = float(np.std(_noise_blocks, ddof=0))
if not np.isfinite(_noise_scale) or _noise_scale <= 0:
    raise RuntimeError('The empirical ground-state noise library is invalid.')
_noise_blocks = _noise_blocks/_noise_scale


def _draw_sim_noise(length):
    pieces = []
    total = 0
    while total < int(length):
        block = _noise_blocks[int(_rng_noise.integers(0, len(_noise_blocks)))]
        if _rng_noise.random() < 0.5:
            block = block[::-1]
        pieces.append(block)
        total += len(block)
    return np.concatenate(pieces)[:int(length)]


# Fraction of the Hann^2 window energy that lies before the decay time.
_sim_window = np.asarray(get_window('hann', int(NPERSEG), fftbins=True), float)
_sim_window_energy = _sim_window*_sim_window
_sim_window_cdf = np.concatenate([
    [0.0], np.cumsum(_sim_window_energy)/np.sum(_sim_window_energy)
])
_sim_window_x = np.linspace(-0.5, 0.5, len(_sim_window_cdf))


def _hann_decay_response(lifetimes):
    lifetimes = np.asarray(lifetimes, float).ravel()
    if len(lifetimes) == 0:
        return np.zeros((0, SIM_NT), float)
    u = (lifetimes[:, None]-SIM_TIME_S[None, :])/SIM_WINDOW_S
    return np.interp(
        u.ravel(), _sim_window_x, _sim_window_cdf, left=0.0, right=1.0
    ).reshape(u.shape)


SIM_N0 = _rng_n0.poisson(SIM_INITIAL_MEAN_IONS_USED, size=SIM_NFILES).astype(int)
SIM_Y = np.empty((SIM_NFILES, SIM_NT), np.float32)
SIM_NOISELESS = np.empty_like(SIM_Y)
SIM_N_EFF = np.empty_like(SIM_Y)
SIM_NOISE_SIGMA = np.empty_like(SIM_Y)
SIM_LIFETIMES_S = []
for _file, _n0 in enumerate(SIM_N0):
    _lifetimes = _rng_tau.exponential(1.0/SIM_RATE_TRUE_S, int(_n0))
    _response = _hann_decay_response(_lifetimes)
    _n_eff = np.sum(_response, axis=0)
    _signal = SIM_A1_TRUE*_n_eff
    _sigma = SIM_A1_TRUE*np.asarray(SIGMA_OF_N(_n_eff), float)
    _noise = _sigma*_draw_sim_noise(SIM_NT)
    SIM_N_EFF[_file] = _n_eff
    SIM_NOISELESS[_file] = _signal
    SIM_NOISE_SIGMA[_file] = _sigma
    SIM_Y[_file] = _signal+_noise
    SIM_LIFETIMES_S.append(_lifetimes)


_integer_mask = np.isclose(SIM_N_EFF, np.rint(SIM_N_EFF), atol=1e-7, rtol=0.0)
if not np.allclose(
        SIM_NOISELESS[_integer_mask],
        SIM_A1_TRUE*np.rint(SIM_N_EFF[_integer_mask]),
        atol=2e-6*max(1.0, SIM_A1_TRUE), rtol=2e-6):
    raise RuntimeError('Internal error: simulation plateaus do not lie exactly at N*A1.')


SIM_FILE_IDX = np.arange(SIM_NFILES, dtype=int)
SIM_MED, SIM_LEN, SIM_ION, SIM_DP_SCORES = fit_integer_dataset(
    SIM_Y, SIM_FILE_IDX
)
SIM_FIT_LEVELS = [A1*np.asarray(state, float) for state in SIM_ION]
if any(np.any(np.diff(level) <= 0) for level in SIM_FIT_LEVELS):
    raise RuntimeError('Internal error: the simulated model staircase is not strictly monotonic.')
SIM_RAW_MEAN_REVERSALS = int(sum(
    np.count_nonzero(np.diff(np.asarray(mean, float)) < 0) for mean in SIM_MED
))

_sim_noise_used = np.asarray(SIM_Y-SIM_NOISELESS, float)
SIM_NOISE_CHECK = {}
_max_check_n = min(NOISE_MAX_N, int(np.max(np.rint(SIM_N_EFF))))
for _n in range(_max_check_n+1):
    _mask = np.isclose(SIM_N_EFF, float(_n), atol=1e-7, rtol=0.0)
    _count = int(np.count_nonzero(_mask))
    if _count >= 20:
        SIM_NOISE_CHECK[_n] = {
            'count': _count,
            'observed_sigma_A1': float(np.std(_sim_noise_used[_mask]/A1, ddof=1)),
            'target_sigma_A1': float(SIGMA_OF_N(float(_n))),
        }

print('='*88)
print('PURE END-TO-END SIMULATION')
print(f'Files={SIM_NFILES} | Input T1/2={SIM_T12_TRUE_MS:.2f} ms '
      f'| dt={1000*SIM_DT_S:.3f} ms | time bins={SIM_NT}')
print(f'Initial ions: target mean={SIM_INITIAL_MEAN_IONS_USED:.3f} '
      f'(real first-bin power/A1={_real_first_bin_mean_ions:.3f}) '
      f'| realized={SIM_N0.mean():.3f} | Median={np.median(SIM_N0):.0f}')
print('Plateau model: noiseless power exactly N*A1; '
      'no single-ion amplitude spread, no random baseline offset.')
print(f'Noise model: sigma(N)/A1=sqrt({SIGMA0_FIT:.3f}^2+'
      f'({K_FIT:.3f}*N)^2), continuous for Hann transition bins')
for _n, _check in SIM_NOISE_CHECK.items():
    print(f'  Generator check N={_n}: sigma={_check["observed_sigma_A1"]:.3f} A1 '
          f'(target {_check["target_sigma_A1"]:.3f}; '
          f'n_bin={_check["count"]})')
print(f'Total simulation noise: mean={_sim_noise_used.mean():+.4f} '
      f'| SD={_sim_noise_used.std(ddof=0):.4f} Power')
print(f'Reconstructed staircase: {sum(len(v) for v in SIM_ION)} segments '
      f'({np.mean([len(v) for v in SIM_ION]):.2f} per file)')
print(f'Monotonicity: model staircase 0 reversals | observed segment means '
      f'{SIM_RAW_MEAN_REVERSALS} reversals')
print('='*88)

simulation_data = {
    'seed': int(SIM_MASTER_SEED),
    'T12_true_ms': float(SIM_T12_TRUE_MS),
    'n_files': int(SIM_NFILES),
    'initial_mean_ions': float(SIM_INITIAL_MEAN_IONS_USED),
    'N0': SIM_N0.copy(),
    'lifetimes_s': SIM_LIFETIMES_S,
    'true_effective_ions': SIM_N_EFF,
    'noise_sigma': SIM_NOISE_SIGMA,
    'noise_calibration': NOISE_CALIBRATION,
    'y': SIM_Y,
    'noiseless': SIM_NOISELESS,
    'observed_segment_means': SIM_MED,
    'fit_medians': SIM_MED,  # compatibility name; these are measured means
    'monotone_fit_levels': SIM_FIT_LEVELS,
    'fit_lengths': SIM_LEN,
    'fit_states': SIM_ION,
}


In [ ]:
# === RESULTS: T1/2 HEATMAPS AND LEVEL-HEIGHT HISTOGRAMS ===

SIM_EVENTS = extract_terminal_10_events(
    SIM_LEN, SIM_ION, SIM_FILE_IDX, SIM_NT
)
if len(SIM_EVENTS) == 0:
    raise RuntimeError('No terminal 1->0 event in the reconstructed simulation.')

T_sim, N_sim, RATE_sim, BOUND_sim, INFO_sim = discrete_matrix(SIM_EVENTS)
BOOT_sim, SIM_BLOCK_FILES_USED = bootstrap_discrete_matrix(
    SIM_EVENTS,
    SIM_FILE_IDX,
    DISC_BOOTSTRAP,
    DISC_BLOCK_FILES_USED,
    DISC_SEED+1,
)
E_sim, CI_sim, NB_sim = bootstrap_summary(BOOT_sim)

_heat_values = np.concatenate([
    T_real[np.isfinite(T_real)], T_sim[np.isfinite(T_sim)]
])
_heat_vmin = float(np.min(_heat_values))
_heat_vmax = float(np.max(_heat_values))
fig, axes = plt.subplots(1, 2, figsize=(17.2, 6.7), constrained_layout=True)
_im_real = draw_discrete_heatmap(
    axes[0], T_real, E_real, N_real,
    'Real data: discrete T1/2 MLE', _heat_vmin, _heat_vmax
)
_im_sim = draw_discrete_heatmap(
    axes[1], T_sim, E_sim, N_sim,
    f'Simulation: reconstructed T1/2 MLE (input {SIM_T12_TRUE_MS:.1f} ms)',
    _heat_vmin, _heat_vmax,
)
fig.colorbar(_im_sim, ax=axes, label=r'$T_{1/2}$ [ms]', shrink=0.88)
plt.show()


SIM_LEVEL_HEIGHTS = np.concatenate([np.asarray(v, float) for v in SIM_MED])
SIM_LEVEL_HEIGHTS = SIM_LEVEL_HEIGHTS[np.isfinite(SIM_LEVEL_HEIGHTS)]
LEVEL_HIST_EDGES = REAL_LEVEL_HIST_EDGES.copy()
fig, ax = plt.subplots(figsize=(11.5, 5.0), constrained_layout=True)
ax.hist(SIM_LEVEL_HEIGHTS, bins=LEVEL_HIST_EDGES,
        color='#35b779', alpha=0.84, edgecolor='none')
for _n in range(max(0, int(np.floor(LEVEL_HIST_EDGES[-1]/A1)))+1):
    ax.axvline(_n*A1, color='black', lw=0.55, alpha=0.22)
ax.set(xlabel='Observed segment mean / power',
       ylabel='Counts (one count per segment)',
       title=f'Simulation: observed segment means ({SIM_NFILES} Files)',
       xlim=(LEVEL_HIST_EDGES[0], LEVEL_HIST_EDGES[-1]))
ax.grid(axis='y', alpha=0.22)
plt.show()

print('='*96)
print('SIMULATED DISCRETE T1/2 ANALYSIS -- NO HALF-BIN')
print(f'Input T1/2={SIM_T12_TRUE_MS:.2f} ms | reconstructed terminal '
      f'1->0 events={len(SIM_EVENTS)} | Bootstrap={DISC_BOOTSTRAP} '
      f'x {SIM_BLOCK_FILES_USED} Files')
print_discrete_matrix('Simulation: T1/2 +/- block-bootstrap SD [ms] (event count)',
                      T_sim, E_sim, N_sim)


_report_L, _report_G = 5, 15
_report_i_l = int(np.flatnonzero(DISC_L_VALUES == _report_L)[0])
_report_i_g = int(np.flatnonzero(DISC_G_VALUES == _report_G)[0])
print(f'\nComparison at L={_report_L}, G={_report_G}: '
      f'real={T_real[_report_i_g, _report_i_l]:.2f} ms '
      f'(n={N_real[_report_i_g, _report_i_l]}), '
      f'sim={T_sim[_report_i_g, _report_i_l]:.2f} ms '
      f'(n={N_sim[_report_i_g, _report_i_l]}), '
      f'simulation input={SIM_T12_TRUE_MS:.2f} ms')
print('\nThe simulation is an end-to-end plausibility test of the staircase fit. '
      'It is not a bias correction for the real data. The decisive question is '
      'whether a common stable L range exists in the real and simulation heatmaps; '
      'NaN fields intentionally contain insufficient statistics.')
print('='*96)

simulation_discrete_results = {
    'T12_input_ms': float(SIM_T12_TRUE_MS),
    'L': DISC_L_VALUES.copy(),
    'G': DISC_G_VALUES.copy(),
    'T12_ms': T_sim,
    'bootstrap_sd_ms': E_sim,
    'bootstrap_ci95_ms': CI_sim,
    'n_events': N_sim,
    'events': SIM_EVENTS.copy(),
    'bootstrap_block_files': int(SIM_BLOCK_FILES_USED),
    'level_heights': SIM_LEVEL_HEIGHTS.copy(),
}
